# LlamaIndex

In [5]:
!pip cache purge --quiet

In [6]:
!pip install openai==1.109.1 \
             llama-index==0.14.23 --quiet
!pip install llama-index-vector-stores-singlestoredb==0.5.0 --no-deps --quiet

In [7]:
import logging
import pandas as pd
import warnings

from llama_index.core import Document, ListIndex, SQLDatabase
from llama_index.core.chat_engine import CondenseQuestionChatEngine
from llama_index.core.indices.vector_store.base import StorageContext, VectorStoreIndex
from llama_index.core.memory import ChatMemoryBuffer
from llama_index.core.query_engine import NLSQLTableQueryEngine
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI
from llama_index.vector_stores.singlestoredb import SingleStoreVectorStore
from singlestoredb.management import get_secret
from sqlalchemy.exc import SAWarning

In [8]:
warnings.filterwarnings("ignore", category = SAWarning)

# Only log errors for key libraries
logging.getLogger("httpx").setLevel(logging.ERROR)
logging.getLogger("llama_index").setLevel(logging.ERROR)
logging.getLogger("sqlalchemy").setLevel(logging.ERROR)

In [9]:
LLM_MODEL = "gpt-4.1-mini"
EMBEDDING_MODEL = "text-embedding-3-small"

os.environ["OPENAI_API_KEY"] = get_secret("OPENAI_API_KEY")

In [10]:
llm = OpenAI(
    model = LLM_MODEL,
    temperature = 0
)

## Natural Language

<div class="alert alert-block alert-warning">
    <b class="fa fa-solid fa-exclamation-circle"></b>
    <div>
        <p><b>Action Required</b></p>
        <p>Select the database from the drop-down menu at the top of this notebook. It updates the <b>connection_url</b> which is used by SQLAlchemy to make connections to the selected database.</p>
    </div>
</div>

In [11]:
from sqlalchemy import *

db_connection = create_engine(connection_url)

In [12]:
db = SQLDatabase(
    db_connection,
    include_tables = ["tick"]
)

In [13]:
sql_query_engine = NLSQLTableQueryEngine(
    sql_database = db,
    tables = ["tick"]
)

In [14]:
print(sql_query_engine.query(
    "Using the 'tick' table, return the ticker symbol with the highest closing price and its close value."
    )
)

The ticker symbol with the highest closing price is WVVF-FX, with a closing price of 6301.59.


In [15]:
print(sql_query_engine.query(
    """
    SELECT symbol, close
    FROM tick
    ORDER BY close DESC
    LIMIT 1;
    """
    )
)

The stock with the highest closing price is WVVF-FX, with a closing price of $6301.59.


## RAG

In [16]:
company_info = {
    "BBRQ-FX": "BBRQ Corp is a global technology conglomerate known for its enterprise software platforms, cloud infrastructure services and a dominant position in business productivity tools.",
    "BJBY-FX": "BJBY Dynamics is a consumer electronics and devices company renowned for its flagship smartphone line, wearable technology and a fast-growing digital services ecosystem.",
    "YWMG-FX": "YWMG Group is an e-commerce and logistics powerhouse with expanding interests in cloud computing, digital advertising and subscription-based media streaming.",
    "HRPD-FX": "HRPD Technologies is a semiconductor and hardware manufacturer supplying chips and processing units to the automotive, aerospace and artificial intelligence industries."
}

In [17]:
# Convert into Documents with metadata
documents = [
    Document(text = text, metadata = {"symbol": symbol})
    for symbol, text in company_info.items()
]

In [18]:
embed_model = OpenAIEmbedding(
    model = EMBEDDING_MODEL
)

<div class="alert alert-block alert-warning">
    <b class="fa fa-solid fa-exclamation-circle"></b>
    <div>
        <p><b>Action Required</b></p>
        <p>Select the database from the drop-down menu at the top of this notebook. It updates the <b>connection_url</b> which is used by SQLAlchemy to make connections to the selected database.</p>
    </div>
</div>

In [19]:
from sqlalchemy import *

db_connection = create_engine(connection_url)

In [20]:
with db_connection.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS company_knowledge;"))

In [21]:
vector_store = SingleStoreVectorStore(
    table_name = "company_knowledge"
)

storage_context = StorageContext.from_defaults(vector_store = vector_store)

index = VectorStoreIndex.from_documents(
    documents,
    storage_context = storage_context,
    embed_model = embed_model,
    llm = llm
)

query_engine = index.as_query_engine()

In [22]:
df = pd.read_sql(
    """
    SELECT
        LEFT(content, 30) AS content,
        LEFT(JSON_ARRAY_UNPACK(vector), 30) AS vector,
        metadata::symbol AS metadata
    FROM company_knowledge
    """,
    db_connection
)

In [23]:
df.head()

,content,vector,metadata
0,YWMG Group is an e-commerce an,"[-0.00719451904,0.0222320557,-",YWMG-FX
1,BBRQ Corp is a global technolo,"[0.0135726929,-0.00692367554,0",BBRQ-FX
2,HRPD Technologies is a semicon,"[-0.0163726807,-0.0491027832,0",HRPD-FX
3,BJBY Dynamics is a consumer el,"[0.0202178955,-0.0174713135,-0",BJBY-FX


In [24]:
print(query_engine.query(
    "What are the most popular consumer devices and services that BBRQ sells?"
).response)

BBRQ Corp is known for its enterprise software platforms, cloud infrastructure services, and business productivity tools.


In [25]:
print(query_engine.query(
    "Describe BJBY's main contributions to both operating systems and cloud services."
).response)

BJBY's main contributions include innovations in operating systems that enhance user experience and efficiency, as well as advancements in cloud services that provide seamless integration and accessibility for users across various devices.


In [26]:
print(query_engine.query(
    "Can you detail the core technologies and services provided by YWMG?"
).response)

YWMG Group offers a range of core technologies and services including e-commerce solutions, logistics services, cloud computing services, digital advertising services, and subscription-based media streaming services.


In [27]:
print(query_engine.query(
    "What are the primary sectors, including e-commerce and cloud, in which HRPD operates?"
).response)

HRPD operates primarily in the automotive, aerospace, and artificial intelligence industries.


## Conversational Memory

In [28]:
memory = ChatMemoryBuffer.from_defaults()

chat_engine = CondenseQuestionChatEngine.from_defaults(
    query_engine = sql_query_engine,
    memory = memory,
    llm = llm
)

In [29]:
print(chat_engine.chat(
    "Show me the last 5 ticks for BBRQ-FX. Present the result as a table."
    )
)

Here are the last 5 ticks for BBRQ-FX:

| Symbol  | Date                | Open   | High    | Low     | Close   | Volume  |
|---------|---------------------|--------|---------|---------|---------|---------|
| BBRQ-FX | 2015-11-24 00:00:00 | 999.83 | 1017.41 | 985.60  | 1004.39 | 2066982 |
| BBRQ-FX | 2015-11-23 00:00:00 | 1020.74| 1039.59 | 1010.85 | 1019.04 | 2005610 |
| BBRQ-FX | 2015-11-20 00:00:00 | 991.68 | 992.87  | 977.93  | 990.32  | 1995583 |
| BBRQ-FX | 2015-11-19 00:00:00 | 1001.18| 1008.44 | 992.92  | 1003.77 | 33279776|
| BBRQ-FX | 2015-11-18 00:00:00 | 1037.09| 1052.68 | 1023.30 | 1042.63 | 5600879 |


In [30]:
print(chat_engine.chat(
    "Now compare BBRQ-FX with BJBY-FX for the same period."
    )
)

Here is a comparison of the last 5 ticks for BBRQ-FX and BJBY-FX for the same dates:

| Symbol   | Date       | Open   | High    | Low     | Close   | Volume  |
|----------|------------|--------|---------|---------|---------|---------|
| BBRQ-FX  | 2015-11-24 | 999.83 | 1017.41 | 985.60  | 1004.39 | 2066982 |
| BJBY-FX  | 2015-11-24 | 423.62 | 427.74  | 419.80  | 423.54  | 4809503 |
| BBRQ-FX  | 2015-11-23 | 1020.74| 1039.59 | 1010.85 | 1019.04 | 2005610 |
| BJBY-FX  | 2015-11-23 | 414.12 | 419.63  | 408.61  | 411.49  | 1094794 |
| BBRQ-FX  | 2015-11-20 | 991.68 | 992.87  | 977.93  | 990.32  | 1995583 |
| BJBY-FX  | 2015-11-20 | 414.89 | 421.62  | 414.55  | 417.29  | 1528386 |
| BBRQ-FX  | 2015-11-19 | 1001.18| 1008.44 | 992.92  | 1003.77 | 33279776|
| BJBY-FX  | 2015-11-19 | 405.89 | 413.27  | 403.93  | 406.81  | 7714812 |
| BBRQ-FX  | 2015-11-18 | 1037.09| 1052.68 | 1023.30 | 1042.63 | 5600879 |
| BJBY-FX  | 2015-11-18 | 392.06 | 394.92  | 385.35  | 392.48  | 4152808 |


In [31]:
print(chat_engine.chat(
    "Which had the higher last close, BBRQ-FX or BJBY-FX?"
    )
)

On the last date in the provided data (2015-11-24), the symbol BBRQ-FX had a higher closing price of 1004.39 compared to BJBY-FX.


In [32]:
print("\nConversation memory buffer (recent):")
for msg in memory.get_all():
    role = msg.role.value.capitalize()
    # Each message may have multiple blocks, we just join text ones
    text = " ".join(
        block.text for block in msg.blocks if hasattr(block, "text")
    )
    print(f"{role}: {text}\n")


Conversation memory buffer (recent):
User: Show me the last 5 ticks for BBRQ-FX. Present the result as a table.

Assistant: Here are the last 5 ticks for BBRQ-FX:

| Symbol  | Date                | Open   | High    | Low     | Close   | Volume  |
|---------|---------------------|--------|---------|---------|---------|---------|
| BBRQ-FX | 2015-11-24 00:00:00 | 999.83 | 1017.41 | 985.60  | 1004.39 | 2066982 |
| BBRQ-FX | 2015-11-23 00:00:00 | 1020.74| 1039.59 | 1010.85 | 1019.04 | 2005610 |
| BBRQ-FX | 2015-11-20 00:00:00 | 991.68 | 992.87  | 977.93  | 990.32  | 1995583 |
| BBRQ-FX | 2015-11-19 00:00:00 | 1001.18| 1008.44 | 992.92  | 1003.77 | 33279776|
| BBRQ-FX | 2015-11-18 00:00:00 | 1037.09| 1052.68 | 1023.30 | 1042.63 | 5600879 |

User: Now compare BBRQ-FX with BJBY-FX for the same period.

Assistant: Here is a comparison of the last 5 ticks for BBRQ-FX and BJBY-FX for the same dates:

| Symbol   | Date       | Open   | High    | Low     | Close   | Volume  |
|----------|--------

In [33]:
# Load sample tick data from SingleStore
df = pd.read_sql(
    """
    SELECT symbol
    FROM tick
    WHERE ts >= (SELECT MAX(ts) FROM tick) - INTERVAL 10 SECOND AND close > 500
    ORDER BY symbol ASC
    """,
    db_connection
)

# Convert each row into a Document
documents = [Document(text = row.to_json()) for _, row in df.iterrows()]

# Create a ListIndex over the JSON documents
json_index = ListIndex.from_documents(documents)

# Convert the DataFrame to a JSON list for deterministic querying
json_data = df.to_dict(orient = "records")

# Deterministic processing to extract tickers and sort alphabetically
tickers = sorted([row["symbol"] for row in json_data])

print("Tickers from last 10 seconds with close > 500:")
print(tickers)

# Query example
query = (
    "Using the latest timestamp in the tick table as the reference, "
    "return all tickers from the 10 seconds before that timestamp "
    "where the close price is above 500 and sort the results alphabetically by ticker."
)

response = json_index.as_query_engine().query(query)

print("\nLLM / Document Query Response (for reasoning / summary purposes):")
print(response.response)

Tickers from last 10 seconds with close > 500:
['BBRQ-FX', 'BBYX-FX', 'BKCZ-FX', 'BMJH-FX', 'BPDZ-FX', 'BPKV-FX', 'BTSP-FX', 'CCMN-FX', 'CGHY-FX', 'CHWP-FX', 'CMJH-FX', 'CPNQ-FX', 'CQNW-FX', 'CRGY-FX', 'CRKV-FX', 'CRYZ-FX', 'CZWQ-FX', 'DBQS-FX', 'DGJR-FX', 'DJMM-FX', 'DJWS-FX', 'DPCV-FX', 'DPHQ-FX', 'DSPZ-FX', 'DWWH-FX', 'DYJN-FX', 'FDPZ-FX', 'FJND-FX', 'FMLH-FX', 'FNDN-FX', 'FPPC-FX', 'FPPT-FX', 'FRGJ-FX', 'FSLV-FX', 'FSQG-FX', 'FTFK-FX', 'FTVP-FX', 'FVMS-FX', 'GDYP-FX', 'GQJT-FX', 'GQVB-FX', 'GTHL-FX', 'GVKX-FX', 'GZKB-FX', 'HBVK-FX', 'HDGG-FX', 'HDTN-FX', 'HHBV-FX', 'HKHY-FX', 'HMWC-FX', 'HWMV-FX', 'HZPN-FX', 'JBPW-FX', 'JJSK-FX', 'JSZL-FX', 'JZPM-FX', 'KBKT-FX', 'KDZD-FX', 'KKVT-FX', 'KLPQ-FX', 'KMYW-FX', 'KQVR-FX', 'KZHV-FX', 'LBCX-FX', 'LCLR-FX', 'LFMD-FX', 'LJDL-FX', 'LKNK-FX', 'LMRZ-FX', 'LQNY-FX', 'MGMX-FX', 'MHHR-FX', 'MHSN-FX', 'MHTD-FX', 'MJBG-FX', 'MJJZ-FX', 'MLSG-FX', 'MNDG-FX', 'MPHX-FX', 'MQRM-FX', 'MSCP-FX', 'MYCD-FX', 'NFYX-FX', 'NMZC-FX', 'NRJZ-FX', 'NVHZ-FX', 'PCZW-